### Setup Environment:

In [ ]:
import os
from src.selfsupervised import train_byol, get_augmentations
from src.byol import BYOL
import pandas as pd

from src.get_dataset import get_dataset, split_data
from src.data_loader import BRSETDataset, process_labels, SSLDataset
from src.model import FoundationalCVModel, FoundationalCVModelWithClassifier
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import DataLoader
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms
import os
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split

# loss function and optimizer
from src.FocalLoss import FocalLoss

# train and test functions
from src.train import train
from src.test import test

In [ ]:
# Constants:
DATASET = '/home/opc/Retina/BRSET/'
DOWNLOAD = False
SHAPE = (224, 224)
IMAGES = os.path.join(DATASET, 'images/')
LABEL = 'DR_ICDR'
TEST_SIZE = 0.2
TEST_SIZE_SSL = 0.4
UNDERSAMPLE = False

LABELS_PATH = os.path.join(DATASET, 'test_ssl.csv')
LABELS_SSL = os.path.join(DATASET, 'train_ssl.csv')
IMAGE_COL = 'image_id'

"""
Dataset Mean and Std:
NORM_MEAN = [0.5896205017400412, 0.29888971649817453, 0.1107679405196557]
NORM_STD = [0.28544273712830986, 0.15905456049750208, 0.07012281660980953]

ImageNet Mean and Std:
NORM_MEAN = [0.485, 0.456, 0.406]
NORM_STD = [0.229, 0.224, 0.225]
"""

NORM_MEAN = None # [0.485, 0.456, 0.406]
NORM_STD = None # [0.229, 0.224, 0.225]

BACKBONE = 'convnextv2_base'
MODE = 'fine_tune'
backbone_mode = 'fine_tune'

HIDDEN = [128]
num_classes = 2

BATCH_SIZE = 16
BATCH_SIZE_SSL = 16
NUM_WORKERS_SSL = 1
NUM_WORKERS = 4

LOSS = None #'focal_loss'
OPTIMIZER = 'adam'

# Define your hyperparameters
num_epochs = 50
learning_rate = 1e-5

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

#### Read csv file:

In [ ]:
def create_ssl_train_test_split(path, download=False, test_size=0.4, random_state=0):
    df = get_dataset(path, download=download, info=False)
    # Split dataset into train, test and validation:
    df_train, df_test = train_test_split(df, test_size=test_size, random_state=random_state)
    df_train.to_csv(os.path.join(path, 'train_ssl.csv'), index=False)
    df_test.to_csv(os.path.join(path, 'test_ssl.csv'), index=False)
    print('Done!')

#create_ssl_train_test_split(DATASET, download=DOWNLOAD, test_size=TEST_SIZE_SSL)    

# Train Self-supervised BYOL model on BRSET

In [ ]:
df = pd.read_csv(LABELS_SSL)
df.head()

In [ ]:
# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Datasets
train_df, eval_df = train_test_split(df, test_size=0.2, random_state=0)

train_dataset = SSLDataset(train_df, IMAGE_COL, IMAGES, shape=SHAPE, transform=get_augmentations(SHAPE))
eval_dataset = SSLDataset(eval_df, IMAGE_COL, IMAGES, shape=SHAPE, transform=get_augmentations(SHAPE))

# Dataloaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE_SSL, shuffle=True, num_workers=NUM_WORKERS_SSL)
eval_loader = DataLoader(eval_dataset, batch_size=8, shuffle=False, num_workers=NUM_WORKERS_SSL)

In [ ]:
# BYOL Model
backbone_model = FoundationalCVModel(backbone=BACKBONE, mode='fine_tune')
byol_model = BYOL(backbone=backbone_model)

byol_model.to(device)

# Use DataParallel to parallelize the model across multiple GPUs
if torch.cuda.device_count() > 1:
    print("Using", torch.cuda.device_count(), "GPUs!")
    model = nn.DataParallel(byol_model, [0,1])

In [ ]:
# Optimizer
optimizer = torch.optim.Adam(byol_model.parameters(), lr=learning_rate)

# Train
train_byol(byol_model, train_loader, eval_loader, num_epochs, optimizer, device, patience=10, path=f'models/checkpoint_{BACKBONE}_byol.pt')

# Downstream task: Image Diabetic Retinopathy Detection

In [ ]:
TEST_SIZE = 0.2
df = pd.read_csv(LABELS_PATH)
# Convert into 2 classes:

# Normal = 0; Non-proliferative = 1, 2, 3; Proliferative = 4
# Map values to categories
df[LABEL] = df[LABEL].apply(lambda x: 'Normal' if x == 0 else 'Diabetic Retinopathy')

In [ ]:
# Split dataset into train, test and validation:
df_train, df_test = split_data(df, LABEL, TEST_SIZE)
print('Getting validation set...')
df_test, df_val = split_data(df_test, LABEL, 0.20)

### Dataloaders

In [ ]:
# Train the one hot encoder on the train set and get the labels for the test and validation sets:
train_labels, mlb, train_columns = process_labels(df_train, col=LABEL)

In [ ]:
# Define the target image shape
SHAPE = (224, 224)  # Adjust to your desired image size

train_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(SHAPE),
    transforms.ToTensor(),
    transforms.RandomHorizontalFlip(),  # Randomly flip the image horizontally
    transforms.RandomRotation(50),  # Randomly rotate the image by up to 10 degrees
])

if NORM_MEAN is not None and NORM_STD is not None:
    train_transforms.transforms.append(transforms.Normalize(mean=NORM_MEAN, std=NORM_STD))

test_transform = transforms.Compose([
    transforms.Resize(SHAPE),
    transforms.ToTensor(),
])

if NORM_MEAN is not None and NORM_STD is not None:
    test_transform.transforms.append(transforms.Normalize(mean=NORM_MEAN, std=NORM_STD))


In [ ]:
# Create the custom dataset
train_dataset = BRSETDataset(
    df_train, 
    IMAGE_COL, 
    IMAGES, 
    LABEL, 
    mlb, 
    train_columns, 
    transform=train_transforms
)

test_dataset = BRSETDataset(
    df_test, 
    IMAGE_COL, 
    IMAGES, 
    LABEL, 
    mlb, 
    train_columns, 
    transform=test_transform
)

val_dataset = BRSETDataset(
    df_val, 
    IMAGE_COL, 
    IMAGES, 
    LABEL, 
    mlb, 
    train_columns, 
    transform=test_transform
)

train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

In [ ]:
# Print 6 samples with their labels
# Iterate through the DataLoader and plot the images with labels
for batch in train_dataloader:
    images, labels = batch['image'], batch['labels']

    for i in range(len(images)):
        if i == 6:
            break
        plt.subplot(2, 3, i + 1)
        plt.imshow(images[i].permute(1, 2, 0))  # Permute to (H, W, C) from (C, H, W)
        plt.title(f"Label: {labels[i]}")
        plt.axis('off')
    plt.show()
    break

### Model

In [ ]:
# Create the model
# backbone_model = FoundationalCVModel(backbone=BACKBONE, mode=MODE)
backbone_model = byol_model.backbone.to("cpu")
model = FoundationalCVModelWithClassifier(backbone_model, hidden=HIDDEN, num_classes=num_classes, mode=MODE, backbone_mode=backbone_mode)
model.to(device)

# Use DataParallel to parallelize the model across multiple GPUs
if torch.cuda.device_count() > 1:
    print("Using", torch.cuda.device_count(), "GPUs!")
    model = nn.DataParallel(model, [0,1])

### Training:

In [ ]:
if LOSS == 'focal_loss':
    class_distribution = train_dataloader.dataset.labels.sum(axis=0)
    print(f'Class distribution: {class_distribution}')
    class_dis = np.array(class_distribution)
    class_weights =1-class_dis/np.sum(class_dis)
    weights = torch.tensor(class_weights).to(device)
    #criterion = FocalLoss()  # Focal Loss
    criterion = FocalLoss(gamma=2, alpha=weights)
else:
    # Assuming train_loader.dataset.labels is a one-hot representation
    class_indices = np.argmax(train_dataloader.dataset.labels, axis=1)

    # Compute class weights using class indices
    class_weights = compute_class_weight('balanced', classes=np.unique(class_indices), y=class_indices)
    class_weights = torch.tensor(class_weights, dtype=torch.float32)
    criterion = nn.CrossEntropyLoss(weight=class_weights).to(device)
    #criterion = nn.BCEWithLogitsLoss() # Binary Cross-Entropy Loss

if OPTIMIZER == 'adam':
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
elif OPTIMIZER == 'adamw':
    optimizer = optim.AdamW(model.parameters(), lr=learning_rate)
else:
    optimizer = optim.SGD(model.parameters(), lr=learning_rate, momentum=0.9)

In [ ]:
model = train(model, train_dataloader, val_dataloader, criterion, optimizer, num_epochs=num_epochs, save=True, device=device, backbone=f'convnextv2_binary_{LABEL}_byol')

### Test

In [ ]:
test(model, test_dataloader, saliency=True, device=device)